# Persisting memory across Strands Agents sessions

In this example you will learn how to persist memory across different sessions in your Strands Agents. 

We will use the use case of an agent that does web search using a `duckduckgo` search API.

In this notebook, we will:
- Explore the capabilities of a memory-powered Strands agent.
- Learn how to store, retrieve, and list memories.
- Understand how to perform web searches via the agent.
- Interact with the agent in an interactive loop.


### Usage Examples

Storing memories:
```
Remember that I prefer tea over coffee
```

Retrieving memories:
```
What do I prefer to drink?
```

Listing all memories:
```
Show me everything you remember about me
```

### Tips for Memory Usage

- Be explicit when asking the agent to remember information
- Use specific queries to retrieve relevant memories
- Memory persistence enables more natural and contextual conversations

## Setup and prerequisites

### Prerequisites
* Python 3.10+
* AWS account and AWS credentials configured in the environment
* Anthropic Claude Sonnet 4.5 enabled on Amazon Bedrock
* IAM role with permissions to create Amazon Bedrock Knowledge Base, Amazon S3 bucket and Amazon DynamoDB

Let's now install the requirement packages for our Strands Agent

In [8]:
# Install the required packages
!pip install -r requirements.txt

In [9]:
%pip show strands-agents strands-agents-tools

Name: strands-agents
Version: 1.61.1
Summary: A model-driven approach to building AI agents in just a few lines of code
Home-page: https://github.com/strands-agents/harness-sdk
Author: 
Author-email: AWS <opensource@amazon.com>
License: Apache-2.0
Location: c:\Sneha\pythoncode\.venv\Lib\site-packages
Requires: boto3, botocore, docstring-parser, httpx, jsonschema, mcp, opentelemetry-api, opentelemetry-instrumentation-threading, opentelemetry-sdk, pydantic, pyyaml, typing-extensions, watchdog
Required-by: strands-agents-tools
---
Name: strands-agents-tools
Version: 0.8.9
Summary: A collection of specialized tools for Strands Agents
Home-page: https://github.com/strands-agents/tools
Author: 
Author-email: AWS <opensource@amazon.com>
License: Apache-2.0
Location: c:\Sneha\pythoncode\.venv\Lib\site-packages
Requires: aiohttp, aws-requests-auth, botocore, dill, markdownify, pillow, prompt-toolkit, pyjwt, requests, rich, slack-bolt, strands-agents, sympy, tenacity, typing-extensions, tzdata, 

In [10]:

# Import Required Libraries
import os
from strands import Agent, tool
#from strands.models import bedrock
#from strands_tools import mem0_memory
from strands_tools.mem0_memory import Mem0MemoryTool
import time

from ddgs import DDGS
from ddgs.exceptions import DDGSException, RatelimitException


os.environ["AWS_REGION"] = os.environ.get("AWS_REGION", "us-east-1")
#bedrock.DEFAULT_BEDROCK_MODEL_ID = "us.anthropic.claude-sonnet-4-5-20250929-v1:0" #Optional: Set a default model for Bedrock

## Mem0 Configuration

### Memory Backend Options

The Mem0 Memory Tool supports three different backend configurations:

1. **[OpenSearch](https://aws.amazon.com/opensearch-service/features/serverless/)** (Recommended for AWS environments):
   - Requires AWS credentials and OpenSearch configuration
   - Set `OPENSEARCH_HOST` and optionally `AWS_REGION` (defaults to us-west-2)

2. **[FAISS]((https://faiss.ai/index.html))** (Default for local development):
   - Uses FAISS as the local vector store backend
   - Requires faiss-cpu package for local vector storage
   - No additional configuration needed

3. **Mem0 Platform**:
   - Uses the [Mem0 Platform API](https://docs.mem0.ai/platform/quickstart) for memory management
   - Requires a Mem0 API key : `MEM0_API_KEY` in the environment variables


### Environment Configuration

| Environment Variable | Description | Default | Required For |
|---------------------|-------------|----------|--------------|
| OPENSEARCH_HOST | OpenSearch Serverless Host URL | None | OpenSearch |
| AWS_REGION | AWS Region for OpenSearch | us-west-2 | OpenSearch |
| MEM0_API_KEY | Mem0 Platform API key | None | Mem0 Platform |
| DEV | Enable development mode | false | All modes |


For the scope of this lab, we can use 2 options as a backend for memory management:
### Option 1. [Opensearch Serverless](https://aws.amazon.com/opensearch-service/features/serverless/) 

This will be our setup architecture for AOSS: 

<div style="text-align:left">
    <img src="images/arch_AOSS.png" width="65%" />
</div>



In [ ]:
# You can manually define your Opensearch Host 
#os.environ["OPENSEARCH_HOST"] = "<your-opensearch-host>.<region>.aoss.amazonaws.com"

In [ ]:
# OR - Run the script to Create Opensearch Serverless resource in your AWS Account
!bash prereqs/deploy_OSS.sh

In [ ]:
wait_time = 120
print(f"Waiting {wait_time} seconds for OpenSearch Serverless collection to be fully operational...")
time.sleep(wait_time)


In [ ]:
# Option 1: Opensearch Serverless
from dotenv import load_dotenv
load_dotenv() # take Opensearch environment variables


### Option 2 [Mem0 Platform](https://docs.mem0.ai/platform):

#### [NOTE]: This is not Needed if you have already deployed the Opensearch Serverless option.

As an alternative, you can create Mem0 API keys by following the steps [here](https://docs.mem0.ai/platform/quickstart#2-api-key-setup) and add it as an environment variable **MEM0_API_KEY**.
This would be the architecture of the setup for Mem0 Platform:

<div style="text-align:left">
    <img src="images/arch_mem0.png" width="65%" />
</div>


To enable the agent's functionality, we need to configure environment variables for AWS credentials and OpenSearch / Mem0 Platform. These variables are used for memory storage and retrieval.

In [ ]:
# Option 2: Mem0 API key

os.environ["MEM0_API_KEY"] = ""
os.environ["MEM0_USER_ID"] = USER_ID

## Define System Prompt

The `SYSTEM_PROMPT` variable defines the behavior and capabilities of the memory agent. This prompt guides the agent to provide personalized responses based on stored memories and perform web searches when necessary.

In [12]:
# Define a focused system prompt for memory operations
SYSTEM_PROMPT = """You are a helpful personal assistant for a user. Your task is to assist the user by providing personalized responses based on their history. 

Capabilities:
- You can store information using the mem0_memory tool (action="store").
- You can retrieve relevant memories using the mem0_memory tool (action="retrieve").
- You can use the websearch tool to find information on the web.

Key Rules:
- Be conversational and natural in your responses.
- Always retrieve memories before responding to the user and use them to inform your response.
- Store any new user information and user preferences in mem0_memory.
- Only share relevant information.
- Politely indicate when you don't have the information.
"""

## Define Web Search Tool

The `websearch` tool using [Duckduckgo Search API](https://github.com/deedy5/duckduckgo_search) function allows the agent to perform web searches. This function handles exceptions and returns search results or appropriate error messages.

In [13]:
@tool
def websearch(
    keywords: str,
    region: str = "us-en",
    max_results: int | None = None,
) -> str:
    """Search the web to get updated information.
    Args:
        keywords (str): The search query keywords.
        region (str): The search region: wt-wt, us-en, uk-en, ru-ru, etc..
        max_results (int | None): The maximum number of results to return.
    Returns:
        List of dictionaries with search results.
    """
    try:
        results = DDGS().text(keywords, region=region, max_results=max_results)
        return results if results else "No results found."
    except RatelimitException:
        return "RatelimitException: Please try again after a short delay."
    except DDGSException as d:
        return f"DuckDuckGoSearchException: {d}"
    except Exception as e:
        return f"Exception: {e}"

## Create Memory Agent

We will now initialize the memory-focused agent using the defined tools and system prompt. The Strands agent is capable of:
1. Storing and retrieving memories based on context. It uses memory to create more personalized and contextual AI interactions.
2. Performing web searches using DuckDuckGo to give updated information.

In [14]:
from strands import Agent
from strands.models.ollama import OllamaModel

model = OllamaModel(
    host="http://localhost:11434",
    model_id="llama3.1",
)



In [15]:
from strands import Agent
from strands_tools.mem0_memory import Mem0MemoryTool

USER_ID = "Sneha"
mem0_tool = Mem0MemoryTool(user_id=USER_ID)

memory_agent = Agent(
    system_prompt=SYSTEM_PROMPT,
    model=model,
    tools=[mem0_tool.mem0_memory, websearch],
)

tool=<<bound method Mem0MemoryTool.mem0_memory of <strands_tools.mem0_memory.Mem0MemoryTool object at 0x00000153222C77D0>>> | unrecognized tool specification


In [ ]:
%pip install --upgrade "strands-agents-tools[mem0_memory]"

## Demonstrate Memory Operations

The following examples demonstrate how to store, retrieve, and list memories using the memory agent.

- **store**: Save important information for later retrieval
  - Store user preferences
  - Remember important facts
  - Maintain conversation context

- **retrieve**: Access relevant memories based on queries
  - Find previously stored information
  - Provide personalized responses based on user history

- **list**: View all stored memories
  - See what information has been retained
  - Audit stored memories

In [18]:
# Store initial memories to demonstrate retrieval
memory_agent.tool.mem0_memory(
    action="store", content=f"The user's name is {USER_ID}.", user_id=USER_ID
)
memory_agent.tool.mem0_memory(
    action="store", 
    content="I like to drink tea more than coffee.", 
    user_id=USER_ID
)

╭───────────────────────────────────────────── Memory for user Sneha ─────────────────────────────────────────────╮
│ The user's name is Sneha.                                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

c:\Sneha\pythoncode\.venv\Lib\site-packages\mem0\client\utils.py:38: DeprecationWarning: output_format='v1.0' is deprecated therefore setting it to 'v1.1' by default. Check out the docs for more information: https://docs.mem0.ai/platform/quickstart#4-1-create-memories
  return func(*args, **kwargs)


╭──────────────────────────────────────────── Memory Operation Error ─────────────────────────────────────────────╮
│ object of type 'NoneType' has no len()                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── Memory for user Sneha ─────────────────────────────────────────────╮
│ I like to drink tea more than coffee.                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── Memory Operation Error ─────────────────────────────────────────────╮
│ object of type 'NoneType' has no len()                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

{'toolUseId': 'tooluse_mem0_memory_476848911',
 'status': 'error',
 'content': [{'text': "Error: object of type 'NoneType' has no len()"}]}

In [19]:
from mem0 import MemoryClient
import inspect

client = MemoryClient()
print(inspect.signature(client.add))

result = client.add(
    [{"role": "user", "content": "I like to drink tea more than coffee."}],
    user_id="Sneha",
    output_format="v1.1",
)

print(result)

(messages: List[Dict[str, str]], **kwargs) -> Dict[str, Any]
{'results': [{'message': 'Memory processing has been queued for background execution', 'status': 'PENDING', 'event_id': '19be502e-b917-408d-8d41-fcea79963616'}], 'relations': []}


In [26]:
stored_memory = memory_agent.tool.mem0_memory(
    action="store",
    content="I like salty food over sweet food"
)

print(stored_memory)

╭───────────────────────────────────────────── Memory for user Sneha ─────────────────────────────────────────────╮
│ I like salty food over sweet food                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── Memory Operation Error ─────────────────────────────────────────────╮
│ object of type 'NoneType' has no len()                                                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

{'toolUseId': 'tooluse_mem0_memory_276034046', 'status': 'error', 'content': [{'text': "Error: object of type 'NoneType' has no len()"}]}


In [21]:
from mem0 import MemoryClient

client = MemoryClient()

result = client.search(
    query="What does Sneha like to drink?",
    user_id="Sneha",
)

print(result)

[{'id': '57a3c4d4-8194-4de4-888f-d244221b07c1', 'memory': "User's name is Sneha.", 'user_id': 'Sneha', 'score': 0.2816, 'categories': ['personal_details'], 'metadata': {}, 'updated_at': '2026-10-16T12:24:19.081405+00:00', 'expiration_date': None, 'created_at': '2026-10-16T12:24:11+00:00', 'structured_attributes': {'day': 16, 'hour': 12, 'year': 2026, 'month': 10, 'minute': 24, 'quarter': 4, 'is_weekend': False, 'day_of_week': 'friday', 'day_of_year': 289, 'week_of_year': 42}}, {'id': '29806fac-2281-48ae-bfe6-ba9a07b8670e', 'memory': 'User prefers drinking tea over coffee', 'user_id': 'Sneha', 'score': 0.1876, 'categories': ['food', 'user_preferences'], 'metadata': {}, 'updated_at': '2026-10-16T12:24:22.318868+00:00', 'expiration_date': None, 'created_at': '2026-10-16T12:24:17+00:00', 'structured_attributes': {'day': 16, 'hour': 12, 'year': 2026, 'month': 10, 'minute': 24, 'quarter': 4, 'is_weekend': False, 'day_of_week': 'friday', 'day_of_year': 289, 'week_of_year': 42}}]


In [27]:
retrieved_memories = memory_agent.tool.mem0_memory(
    action="retrieve",
    query="What does Sneha like, salty or sweet food?"
)

print(retrieved_memories)

╭──────────────────────────────────────────────── Search Results ─────────────────────────────────────────────────╮
│                                                 Search Results                                                  │
│ ┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┓ │
│ ┃           ┃                                                    ┃           ┃ Created   ┃         ┃          ┃ │
│ ┃ ID        ┃ Memory                                             ┃ Relevance ┃ At        ┃ User ID ┃ Metadata ┃ │
│ ┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━┩ │
│ │ 57a3c4d4… │ User's name is Sneha.                              │ 0.2489    │ 2026-10-… │ Sneha   │ None     │ │
│ │ 86438617… │ User prefers salty food over sweet food            │ 0.2479    │ 2026-10-… │ Sneha   │ None     │ │
│ │ 29806fac… │ User prefers drinking tea over coffee              │ 0.1496    │ 2026-10-… │ Sneha   │ None     │ │
│ └───────────┴────────────────────────────────────────────────────┴───────────┴───────────┴─────────┴──────────┘ │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

{'toolUseId': 'tooluse_mem0_memory_677101455', 'status': 'success', 'content': [{'text': '[\n  {\n    "id": "57a3c4d4-8194-4de4-888f-d244221b07c1",\n    "memory": "User\'s name is Sneha.",\n    "user_id": "Sneha",\n    "score": 0.2489,\n    "categories": [\n      "personal_details"\n    ],\n    "metadata": {},\n    "updated_at": "2026-10-16T12:24:19.081405+00:00",\n    "expiration_date": null,\n    "created_at": "2026-10-16T12:24:11+00:00",\n    "structured_attributes": {\n      "day": 16,\n      "hour": 12,\n      "year": 2026,\n      "month": 10,\n      "minute": 24,\n      "quarter": 4,\n      "is_weekend": false,\n      "day_of_week": "friday",\n      "day_of_year": 289,\n      "week_of_year": 42\n    }\n  },\n  {\n    "id": "86438617-26a4-4c52-89aa-42b34da9694e",\n    "memory": "User prefers salty food over sweet food",\n    "user_id": "Sneha",\n    "score": 0.2479,\n    "categories": [\n      "food",\n      "user_preferences"\n    ],\n    "metadata": {},\n    "updated_at": "2026-

In [ ]:
import os

print("MEM0_API_KEY configured:", bool(os.getenv("MEM0_API_KEY")))
print("MEM0_USER_ID:", os.getenv("MEM0_USER_ID"))

In [ ]:
# Retrieve memories
retrieved_memories = memory_agent.tool.mem0_memory(
    action="retrieve", query="What is the user's name?", user_id=USER_ID
)
print("Retrieved Memories:", retrieved_memories)

In [ ]:
# Retrieve memories about preferences
memory_agent.tool.mem0_memory(
    action="retrieve",
    query="What are the my drink preferences?",
    user_id=USER_ID
)

In [23]:
# Ask the agent a question
response = memory_agent("What are the events happening in the New York today?")
print(response)


Tool #1: websearch
Based on the web search results, there are several events happening in New York today. Some of the events include:

* Events and Things to do in New York, NY today - Eventbrite
* Discover Events Today Events & Activities in New York, NY | Eventbrite
* Best Things to do in NYC Today with Free Events, Concerts and Art
* NYC Events Today
* Things to Do in NYC Today: Friday, Oct 16, 2026
* Events Today in NYC - Happening NYC
* Free in NYC — Free NYC Events Every Day
* New York Events Calendar | Happenings Today & This Weekend

These events include a variety of activities such as concerts, comedy, gallery nights, and last-chance runs. Some of the events are free, while others may require a ticket or RSVP. It's always a good idea to check the event details and schedule before heading out.Based on the web search results, there are several events happening in New York today. Some of the events include:

* Events and Things to do in New York, NY today - Eventbrite
* Discover

In [25]:
response = memory_agent("What do you remember about my drink preference?")
print(response)


Tool #2: mem0_memory


╭──────────────────────────────────────────────── Search Results ─────────────────────────────────────────────────╮
│                                                 Search Results                                                  │
│ ┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┓ │
│ ┃           ┃                                                    ┃           ┃ Created   ┃         ┃          ┃ │
│ ┃ ID        ┃ Memory                                             ┃ Relevance ┃ At        ┃ User ID ┃ Metadata ┃ │
│ ┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━┩ │
│ │ 29806fac… │ User prefers drinking tea over coffee              │ 0.2863    │ 2026-10-… │ Sneha   │ None     │ │
│ └───────────┴────────────────────────────────────────────────────┴───────────┴───────────┴─────────┴──────────┘ │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

You prefer drinking tea over coffee.You prefer drinking tea over coffee.



In [24]:

# List all stored memories
print("All Stored Memories:")
all_memories = memory_agent.tool.mem0_memory(
    action="list", user_id=USER_ID
)

All Stored Memories:


╭───────────────────────────────────────────────── Memories List ─────────────────────────────────────────────────╮
│                                                    Memories                                                     │
│ ┏━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━━━━┓ │
│ ┃ ID              ┃ Memory                                             ┃ Created At      ┃ User ID ┃ Metadata ┃ │
│ ┡━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━━━━┩ │
│ │ 29806fac-2281-… │ User prefers drinking tea over coffee              │ 2026-10-16T05:… │ Sneha   │ None     │ │
│ │ 57a3c4d4-8194-… │ User's name is Sneha.                              │ 2026-10-16T05:… │ Sneha   │ None     │ │
│ └─────────────────┴────────────────────────────────────────────────────┴─────────────────┴─────────┴──────────┘ │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Interactive Agent Usage

Finally, we provide an interactive loop for users to interact with the memory agent. Users can store new memories, retrieve existing ones, or list all stored memories.

To test interactive usage: 
1. Install the requirements: `pip install -r requirements.txt`
1. Run the python file `personal_agent_with_memory.py`. 

## Conclusion

This notebook demonstrates how to create a personal agent with memory capabilities using the Strands framework. The agent can:

1. Store information about the user
2. Retrieve relevant memories based on context
3. Search the web for additional information
4. Provide personalized responses

By combining these capabilities, the agent can maintain context across conversations and provide more personalized assistance over time.

### Cleanup

Run this bash script to clean up the Opensearch Serverless resources. You don't need to run this if you used the "MEM0_PLATFORM_API".

In [ ]:
!sh prereqs/cleanup_OSS.sh